# 🗳️ Notebook 1 — Quorum Basics: From Single Node to W + R > N

Welcome! In this lab we build a tiny **replicated key–value store** from scratch
and discover the rule that powers Amazon DynamoDB, Apache Cassandra, and Riak:

> **If `W + R > N` then every read overlaps with the latest write → strong consistency.**

We'll walk a **bad → better → best** progression:

| Step | Replication | Consistency | Availability |
|------|-------------|-------------|--------------|
| ❌ 1. Single node | none | fine (1 copy) | **dies** if node dies |
| ⚠️ 2. Replicated, `W=R=1` | N copies | **stale reads** possible | very high |
| ✅ 3. Quorum `W+R>N` | N copies | **strong** | slightly lower |

No prior distributed-systems knowledge required — if you know Python lists and
dictionaries, you're good.


## 🛠️ Setup

From a terminal, in `02-distributed-primitives/quorum`:

```bash
uv sync
```

Then in VS Code pick the `.venv` kernel (top-right of this notebook). If it
doesn't appear, `Cmd+Shift+P` → **Reload Window**.


## 🧠 The intuition (pigeonhole principle)

Imagine **N = 5** replicas drawn as 5 boxes.

- A write with **W = 3** must touch **3** of them.
- A read with **R = 3** must touch **3** of them.

Can you pick two sets of 3 boxes out of 5 that **don't share at least one box**?
No — there are only 5 boxes, so 3 + 3 = 6 pigeons into 5 holes means **at least
one box is in both sets**. That shared box is the one holding the latest value,
and since we ask every reader for a timestamp and take the newest, we're safe.

That's the whole trick: **`W + R > N` forces an overlap**.


## ❌ Step 1 — A single node (no replication)

The simplest "database" is a Python dict. Works great… until the process dies.


In [ ]:
class SingleNodeStore:
    def __init__(self):
        self.data = {}
        self.alive = True

    def write(self, k, v):
        if not self.alive:
            raise RuntimeError("node is down!")
        self.data[k] = v

    def read(self, k):
        if not self.alive:
            raise RuntimeError("node is down!")
        return self.data.get(k)

store = SingleNodeStore()
store.write("user:42", "Ada")
print("read:", store.read("user:42"))
assert store.read("user:42") == "Ada"

# now the node crashes
store.alive = False
try:
    store.read("user:42")
except RuntimeError as e:
    print("💥", e)
else:
    raise AssertionError("a dead node should not answer")
# Availability drops to zero, not "degraded": there is no other copy to fall back to.
assert store.data, "the data still exists — we just cannot reach it"



**Lesson:** a single node is a single point of failure. Let's replicate the
data across several machines.

## 🧱 A replicated cluster

Each replica keeps its own copy of `(value, timestamp)` per key. Writes fan out
to **every** replica; reads contact `R` of them and pick the one with the
**highest timestamp** (last-write-wins conflict resolution).


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple
import random

@dataclass
class Replica:
    name: str
    up: bool = True
    data: Dict[str, Tuple[str, int]] = field(default_factory=dict)

    def write(self, k, v, ts):
        if not self.up:
            return False                       # can't reach a dead replica
        cur = self.data.get(k)
        if cur is None or ts > cur[1]:         # keep freshest by timestamp
            self.data[k] = (v, ts)
        return True

    def read(self, k):
        if not self.up:
            return None
        return self.data.get(k)                # (value, ts) or None


class Cluster:
    def __init__(self, n=5, seed=0):
        self.replicas = [Replica(f"r{i}") for i in range(n)]
        self.n = n
        self._ts = 0                           # central clock for simplicity
        self.rng = random.Random(seed)

    def _next_ts(self):
        self._ts += 1
        return self._ts

    def write(self, key, value, W):
        '''Send write to every replica. Succeed if >= W acknowledge.'''
        ts = self._next_ts()
        acks = sum(1 for r in self.replicas if r.write(key, value, ts))
        return (acks >= W), ts, acks

    def write_to(self, key, value, names):
        '''Force the write onto exactly this set of replicas.

        A real coordinator fans out to all N and returns as soon as W ack; which W
        got there first is up to the network. To *study* the W+R rule we need to pin
        the write set down, so this helper lets us choose it.'''
        ts = self._next_ts()
        for r in self.replicas:
            if r.name in names:
                r.write(key, value, ts)
        return ts

    def read(self, key, R, from_names=None):
        '''Contact R replicas (a random R unless pinned) and return the freshest reply.

        We sample at random rather than always taking the first R in list order —
        otherwise the "quorum" would silently always be the same replicas and the
        experiments below would prove nothing.'''
        live = [r for r in self.replicas if r.up]
        if from_names is not None:
            chosen = [r for r in self.replicas if r.name in from_names and r.up]
        else:
            if len(live) < R:
                return None                    # not enough replicas -> fail the read
            chosen = self.rng.sample(live, R)
        if len(chosen) < R:
            return None
        responses = [(r.name, r.read(key)) for r in chosen]
        with_data = [x for x in responses if x[1] is not None]
        if not with_data:
            return None
        return max(with_data, key=lambda x: x[1][1])   # freshest ts wins


## ⚠️ Step 2 — Replicated but W = R = 1 (eventual consistency)

High availability: any single live replica can answer. But a slow/partitioned
replica can return **stale** data.

**Scenario:**
1. Write `v1` to all 5 replicas (everyone is in sync).
2. A network partition isolates `r1..r4`. Only `r0` is reachable.
3. Client writes `v2` with `W=1` — only `r0` gets it.
4. Partition heals. Client reads with `R=1` from a random replica.

Roughly **4 out of 5** reads will return the **stale** `v1`.


In [ ]:
c = Cluster(n=5, seed=42)
c.write("x", "v1", W=5)                        # everyone has v1

# partition: r1..r4 unreachable
for r in c.replicas[1:]:
    r.up = False
c.write("x", "v2", W=1)                        # only r0 is updated

# heal
for r in c.replicas:
    r.up = True

stale = fresh = 0
for _ in range(1000):
    value = c.read("x", R=1)[1][0]
    if value == "v1":
        stale += 1
    else:
        fresh += 1
print(f"R=1 reads over 1000 tries → fresh={fresh}, stale={stale}")

# Only r0 has v2, so a uniformly random single-replica read is stale 4 times in 5.
assert stale + fresh == 1000
assert 0.75 < stale / 1000 < 0.85, stale / 1000
print(f"That's {stale/10:.0f}% stale reads — clearly not OK for e.g. a bank balance.")


## ✅ Step 3 — Quorum: W + R > N (strong consistency)

Same partition scenario, but now we write with `W=3` and read with `R=3`.
`3 + 3 = 6 > 5 = N`, so every read must **overlap with the write set** and
therefore sees `v2`.


In [ ]:
c = Cluster(n=5, seed=7)
c.write("x", "v1", W=5)

# partition: r1 and r2 are unreachable
c.replicas[1].up = False
c.replicas[2].up = False
ok, ts, acks = c.write("x", "v2", W=3)
print(f"write v2 with W=3 → ok={ok}, acks={acks}  (landed on r0, r3, r4)")
assert ok and acks == 3

# heal
for r in c.replicas:
    r.up = True

for _ in range(5):
    print("R=3 read →", c.read("x", R=3))

# Five samples is not evidence — every possible quorum must see v2, so try a lot.
assert all(c.read("x", R=3)[1][0] == "v2" for _ in range(2000))
print("\n✔ 2000/2000 randomly-chosen R=3 quorums returned v2")


Every read returns `v2`. That's the W + R > N guarantee in action.

## 🔬 Why "every read" and not "almost every read"

Two thousand successful samples is evidence, not a proof — and the near-miss case is the one
worth being sure about. `W=R=2` with `N=5` gives `W+R=4`, only *one short* of the rule. Systems
get configured that way by accident all the time, and they look fine until they don't.

`N=5` is small enough to settle by brute force: enumerate **every** write set of size `W`
against **every** read set of size `R` and count the pairs that share no replica. A disjoint
pair is a concrete stale read waiting to happen.

In [ ]:
from itertools import combinations

def disjoint_pairs(n, W, R):
    """Every (write-set, read-set) pair sharing no replica = a possible stale read."""
    nodes = range(n)
    return [(w, r) for w in combinations(nodes, W)
                   for r in combinations(nodes, R)
                   if not set(w) & set(r)]

N = 5
print(f"{'W':>2} {'R':>2} {'W+R':>4} {'>N?':>6} {'disjoint (W-set, R-set) pairs':>31}")
print("-" * 48)
for W in range(1, N + 1):
    for R in range(1, N + 1):
        bad = disjoint_pairs(N, W, R)
        overlaps = W + R > N
        print(f"{W:>2} {R:>2} {W+R:>4} {str(overlaps):>6} {len(bad):>31}")
        # This IS the theorem, checked rather than quoted:
        # guaranteed overlap  <=>  W + R > N.
        assert (len(bad) == 0) == overlaps, (W, R, len(bad))

print("\n✔ zero disjoint pairs for every W+R > N, and at least one for every W+R <= N")

### Now run the counterexample the table found

`W=2, R=2` on `N=5` has disjoint pairs. Pick one — write to `{r0, r1}`, read from `{r2, r3}` —
and watch the read miss the write entirely. Same cluster, same code, one config change away
from the correct setup above.

In [ ]:
c = Cluster(n=5, seed=1)
c.write("y", "v1", W=5)                      # everyone starts in sync

w_set, r_set = disjoint_pairs(5, 2, 2)[0]
w_names = {f"r{i}" for i in w_set}
r_names = {f"r{i}" for i in r_set}
print(f"write set {sorted(w_names)}, read set {sorted(r_names)} — no overlap")

c.write_to("y", "v2", w_names)               # W=2 replicas ack, the write returns success
value, ts = c.read("y", R=2, from_names=r_names)[1]

print(f"client wrote v2, then read back: {value!r}")
assert value == "v1", "expected the stale value"
print("\n💥 the write succeeded, the read succeeded, and the client got last week's data.")
print("   Nothing errored anywhere. That is what W+R <= N actually costs you.")

# Same cluster, same (disjoint) read set, W bumped to 4 so that W+R = 6 > 5.
c2 = Cluster(n=5, seed=1)
c2.write("y", "v1", W=5)
c2.write_to("y", "v2", {f"r{i}" for i in range(4)})     # W=4
value2, _ = c2.read("y", R=2, from_names=r_names)[1]
assert value2 == "v2", value2
print(f"\n✔ with W=4, R=2 (sum 6 > 5) that very same read set returns {value2!r}")

## 📊 Cheat sheet (N = 5)

| W | R | W+R | Guarantee        | Good for                                  |
|---|---|-----|------------------|-------------------------------------------|
| 5 | 1 |  6  | strong           | read-heavy; can tolerate write outages    |
| 1 | 5 |  6  | strong           | write-heavy logs; reads can be slow       |
| 3 | 3 |  6  | strong, balanced | typical OLTP                              |
| 2 | 2 |  4  | **eventual**     | fast everything, staleness OK             |
| 1 | 1 |  2  | **eventual**     | max availability (DNS, shopping carts)    |

Picking `W` and `R` is the main **dial** in Dynamo-style databases between
consistency, availability, and latency.


## 🎯 Try it yourself

1. Set `N=7`, `W=4`, `R=4`. Partition any 3 replicas. Do reads still see the
   latest write? Check it with `disjoint_pairs(7, 4, 4)`.
2. Run `disjoint_pairs(5, 3, 2)` — `W+R = 5`, exactly equal to `N`. How many stale-read
   configurations are there? (The rule is strict: `>`, not `≥`.)
3. `W=1, R=5` also satisfies `W+R>N`. What breaks about it that `W=3, R=3` does not?
4. What happens to write availability as `W` grows toward `N`? We'll quantify
   this in **Notebook 2**.
